## Unstructured data to Vector Embeddings

![](./images/preparing_data.png)


## Initialize

In [ ]:
import os

import azure.identity
import dotenv
import openai

# Set up OpenAI client based on environment variables

dotenv.load_dotenv()
print("Loaded environment variables from .env file")
print(dotenv.dotenv_values(".env"))
AZURE_OPENAI_SERVICE = os.getenv("AZURE_OPENAI_SERVICE")
AZURE_OPENAI_EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")
AZURE_OPENAI_TEXT_COMPLETION_MODEL = os.getenv("AZURE_OPENAI_TEXT_COMPLETION_MODEL")

azure_credential = azure.identity.AzureDeveloperCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID"))
token_provider = azure.identity.get_bearer_token_provider(
    azure_credential, "https://cognitiveservices.azure.com/.default"
)
openai_client = openai.AzureOpenAI(
    api_version="2024-06-01",
    azure_endpoint=f"https://{AZURE_OPENAI_SERVICE}.openai.azure.com",
    azure_ad_token_provider=token_provider,
)

### Breaking down data in smaller chunks


In [ ]:
import json

import numpy as np

from utils import format_movie

# Load movie data
with open("./movies.json", "r") as file:
    data = json.load(file)

# Step 1: Extract movie details
movies = data["movies"]

# Step 2: Create chunks of movie details
chunk_size = 1  # Number of movies per chunk
chunks = []

for i in range(0, len(movies), chunk_size):
    chunk = movies[i : i + chunk_size]
    # format each movie into text and join with double newline if chunk_size > 1
    chunk_text = "\n\n".join(format_movie(movie) for movie in chunk)
    chunks.append(chunk_text)

# Step 3: Display the chunks
print("Chunked Movie Details:")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i + 1}:\n{chunk}\n")

### Creating Embeddings


In [ ]:
chunk_embeddings = []
for chunk_text in chunks:
    response = openai_client.embeddings.create(model=AZURE_OPENAI_EMBEDDING_MODEL, input=chunk_text)
    embedding = response.data[0].embedding
    chunk_embeddings.append(embedding)

print("Chunk embeddings created:", len(chunk_embeddings), "chunks.")
print("First chunk embedding:", chunk_embeddings[0][:10])

### Why is length of all Embeddings = 1536?


In [ ]:
# chunk_embeddings[1]
len(chunk_embeddings[1])

# Storing the Data


### Indexing


In [ ]:
import faiss

# Step 4: Index the embeddings using FAISS
dimension = len(chunk_embeddings[0])  # Dimensionality of the embeddings
index = faiss.IndexFlatL2(dimension)  # Create a flat (non-compressed) index

# Normalize embeddings first
embeddings_array = np.array(chunk_embeddings).astype("float32")
faiss.normalize_L2(embeddings_array)

# Build an IP index (inner product)
dimension = embeddings_array.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings_array)

faiss.write_index(index, "movie_title_embeddings_cosine.index")